<img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcSEaxoapR7GyB6FC7xp4aM-SrjhxTWTdFCmKAX3TvvFvQ&s=10" alt="Tecnológico de Costa Rica" width="160">

# 🌎 WALC 2026
### Ciencia de Datos · Introducción al Aprendizaje Automático
**Día 4 (Jueves) — No supervisado — Agrupación (K-means) y asociación (Apriori)**

**Instituto Tecnológico de Costa Rica (TEC)** · Instructor: **Efrén Jiménez Delgado**
· 📧 efjimenez@itcr.ac.cr · 🔗 [LinkedIn](https://www.linkedin.com/in/ejimenezdelgado/)

# 🛍️ Reglas de asociación: Apriori, Eclat y FP-Growth con compras en línea (Instacart)

## 🧵 El hilo conductor (metodología CRISP-DM)

Este cuaderno sigue las seis fases de **CRISP-DM** [1] [2]: cada título de fase marca dónde estamos en el proceso.

| Fase | En este cuaderno |
|---|---|
| 🎯 Comprensión del negocio | Descubrir qué productos se compran juntos en un servicio de compras de víveres en línea. |
| 🔍 Comprensión de los datos | Descargamos pedidos y catálogo de productos (Instacart) y tomamos los primeros 10 000 pedidos. |
| 🧹 Preparación de los datos | Unimos pedidos con nombres de producto, conservamos los productos con al menos 0.5 % de los pedidos y codificamos como matriz verdadero/falso. |
| 🤖 Modelado | Aplicamos Apriori y FP-Growth; implementamos Eclat. |
| 📊 Evaluación | Interpretamos las reglas y comparamos los tiempos de los algoritmos. |
| 🚀 Despliegue y conclusiones | Conclusiones. |

> 🎯 **Analogía:** un asistente de compras en línea que, mirando miles de carritos, aprende qué productos se agregan juntos y sugiere "¿olvidó algo?" antes de pagar.

### 🗺️ Mapa del curso (cronograma Track 5 · Ciencia de datos)
| Día | Jornada | Teoría y laboratorio |
|---|---|---|
| 1 | Lunes | Predicción: regresión lineal y series de tiempo |
| 2 | Martes | Clasificación 1: árboles de decisión, bosques aleatorios y KNN |
| 3 | Miércoles | Clasificación 2: SVM, regresión logística, Naive Bayes y redes neuronales |
| 👉 **4** | Jueves | **No supervisado: agrupación y asociación** |
| 5 | Viernes | Ciencia de datos con IA generativa, ética y buenas prácticas |

# Reglas de asociación: Apriori, Eclat y FP-Growth con compras en línea

El análisis de reglas de asociación es una técnica clave del comercio minorista para descubrir qué artículos se compran juntos [4]. Aquí se aplica a pedidos de una tienda de víveres en línea y se comparan tres algoritmos.

### Autores
* Efrén Jiménez

---
## 🎯 Fase 1 · Comprensión del negocio

Descubrir qué productos se compran juntos en un servicio de compras de víveres en línea.

📚 Lectura recomendada: [3]

## Análisis del problema

**Pregunta de negocio:** ¿qué productos suelen comprarse juntos en un pedido en línea? Las reglas permiten sugerir productos antes del pago o armar paquetes [3].

### Análisis de la canasta de mercado

El *market basket analysis* estudia qué artículos coexisten en una misma transacción (aquí, un pedido). Aplicaciones: venta cruzada, recomendaciones y ubicación de productos [9].

### 🧪 Teoría rápida: reglas de asociación

Una **regla de asociación** tiene la forma **A → B** ("quien compra A suele comprar también B") y se descubre en datos de **transacciones** (canastas de compra) [4]. Se evalúa con tres medidas:

| Medida | Fórmula | Se lee como |
|---|---|---|
| **Soporte** de A | $\text{sop}(A)=\dfrac{\#\{\text{transacciones que contienen }A\}}{\#\{\text{transacciones}\}}$ | qué tan frecuente es el conjunto |
| **Confianza** de A → B | $\text{conf}(A\to B)=\dfrac{\text{sop}(A\cup B)}{\text{sop}(A)}$ | probabilidad de B dado A |
| **Elevación** (*lift*) | $\text{lift}(A\to B)=\dfrac{\text{conf}(A\to B)}{\text{sop}(B)}$ | cuántas veces más probable es B si hay A, frente a B sin saber nada |

Un *lift* > 1 indica asociación positiva, = 1 independencia y < 1 que se **sustituyen** [6]. Otras medidas útiles son el **apalancamiento** (*leverage*) $\text{sop}(A\cup B)-\text{sop}(A)\,\text{sop}(B)$ y la **convicción** $\dfrac{1-\text{sop}(B)}{1-\text{conf}(A\to B)}$ [10].

**Algoritmo Apriori.** Se apoya en una propiedad: *todo subconjunto de un conjunto frecuente también es frecuente* (y, al revés, si un conjunto no es frecuente, ninguno de sus superconjuntos lo es), lo que permite **podar** candidatos [5]. El proceso tiene dos pasos: (1) hallar los **conjuntos frecuentes** con soporte ≥ `min_support`; (2) generar las **reglas** con confianza (o lift) ≥ umbral [9]. Alternativas más rápidas son **FP-Growth** [7] y **Eclat** [8].

In [ ]:
%pip install -q squarify mlxtend

In [ ]:
# Manipulación de datos
import pandas as pd
import numpy as np

# Visualización
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import squarify
sns.set_style('whitegrid')

# Reglas de asociación (mlxtend)
from mlxtend.frequent_patterns import apriori, association_rules, fpgrowth
from mlxtend.preprocessing import TransactionEncoder

---
## 🔍 Fase 2 · Comprensión de los datos

Descargamos pedidos y catálogo de productos (Instacart) y tomamos los primeros 10 000 pedidos.

📚 Lectura recomendada: [3]

In [ ]:
import os, io, zipfile, urllib.request

def descargar(repo, interno_zip, salida):
    """Baja el zip del repositorio de GitLab y extrae un zip interno."""
    if os.path.exists(salida): return
    z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(f'https://gitlab.com/ejimenez/{repo}/-/archive/main/{repo}-main.zip').read()))
    nombre = [n for n in z.namelist() if n.endswith(interno_zip)][0]
    zipfile.ZipFile(io.BytesIO(z.read(nombre))).extractall('.')
    for raiz, _, archivos in os.walk('.'):          # por si el CSV queda dentro de una carpeta
        if salida in archivos and raiz != '.':
            os.replace(os.path.join(raiz, salida), salida)

descargar('productsordersdataset', 'order_products__prior.zip', 'order_products__prior.csv')
descargar('productsordersdataset', 'products.csv.zip', 'products.csv')
print([f for f in ('order_products__prior.csv', 'products.csv') if os.path.exists(f)])

In [ ]:
# Se usan los primeros 10 000 pedidos completos (el archivo completo tiene millones de filas)
orders = pd.read_csv('order_products__prior.csv')
primeros = orders['order_id'].drop_duplicates().iloc[:10000]
orders = orders[orders['order_id'].isin(primeros)]
print('Filas (producto en pedido):', orders.shape[0], '| pedidos:', orders['order_id'].nunique())

In [ ]:
orders.head()

In [ ]:
products = pd.read_csv('products.csv')
print('Catálogo:', products.shape[0], 'productos')

---
## 🧹 Fase 3 · Preparación de los datos

Unimos pedidos con nombres de producto, conservamos los productos con al menos 0.5 % de los pedidos y codificamos como matriz verdadero/falso.

📚 Lectura recomendada: [11]

In [ ]:
# Unir cada línea de pedido con el nombre del producto y quedarse con lo necesario
data = pd.merge(orders, products, on='product_id', how='left')[['order_id', 'product_name']]
data.head()

In [ ]:
# Cada pedido es una transacción: lista de nombres de producto
canasta = data.groupby('order_id')['product_name'].apply(list)

# Podar: un producto con soporte < 0.5 % jamás aparecerá en un conjunto frecuente con min_support = 0.005,
# así que se descarta antes de codificar (evita una matriz enorme con miles de columnas casi vacías)
conteo = data['product_name'].value_counts()
frecuentes = set(conteo[conteo >= 0.005 * len(canasta)].index)
canasta = canasta.apply(lambda lista: [p for p in lista if p in frecuentes])

t = TransactionEncoder()
data = pd.DataFrame(t.fit_transform(canasta.tolist()), columns=t.columns_)
print('Matriz pedidos × productos:', data.shape)

### Preparación de los datos

La versión original armaba una tabla ancha con `pivot_table` sobre los 100 000 primeros *renglones* (pedidos cortados a la mitad y miles de columnas). Aquí se usan pedidos completos, se **podan** los productos poco frecuentes y se codifica con `TransactionEncoder` [11].

In [ ]:
r = data.sum(axis=0).sort_values(ascending=False)[:20]
print(f'Producto más frecuente: {r.index[0]} ({r.iloc[0] / len(data):.1%} de los pedidos)')
plt.figure(figsize=(12, 5))
sns.barplot(x=r.index, y=r.values, color='steelblue')
plt.xticks(rotation=90)
plt.ylabel('Número de pedidos')
plt.title('20 productos más pedidos')
plt.tight_layout()
plt.show()

In [ ]:
cmap = matplotlib.cm.Blues
norm = matplotlib.colors.Normalize(vmin=r.values.min(), vmax=r.values.max())
colors = [cmap(norm(v)) for v in r.values]
plt.figure(figsize=(10, 7))
squarify.plot(sizes=r.values, label=r.index, alpha=.7, color=colors, text_kwargs={'fontsize': 8})
plt.title('Mapa de árbol de los 20 productos más pedidos')
plt.axis('off')
plt.show()

---
## 🤖 Fase 4 · Modelado

Aplicamos Apriori y FP-Growth; implementamos Eclat.

📚 Lectura recomendada: [5] [7] [8]

### Algoritmo Apriori

Buscamos los conjuntos de productos presentes en **al menos el 0.5 %** de los pedidos (`min_support = 0.005`).

In [ ]:
freq_items = apriori(data, min_support=0.005, use_colnames=True)
freq_items['longitud'] = freq_items['itemsets'].apply(len)
print('Conjuntos frecuentes:', len(freq_items))
print(freq_items['longitud'].value_counts().sort_index().rename('conjuntos').to_frame())

In [ ]:
freq_items.sort_values('support', ascending=False).head(10)

---
## 📊 Fase 5 · Evaluación

Interpretamos las reglas y comparamos los tiempos de los algoritmos.

📚 Lectura recomendada: [6] [10]

In [ ]:
# Reglas con confianza >= 20 % y asociación positiva (lift > 1)
res = association_rules(freq_items, metric='confidence', min_threshold=0.2)
res = res[res['lift'] > 1].sort_values('lift', ascending=False)
print('Reglas generadas:', len(res))

In [ ]:
res[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10)

# Lectura automática de las 3 reglas con mayor lift
for _, f in res.head(3).iterrows():
    print(f"• El {f.confidence:.0%} de los pedidos con {' + '.join(sorted(f.antecedents))} también contienen "
          f"{' + '.join(sorted(f.consequents))} (soporte {f.support:.1%}, lift {f.lift:.2f}).")

## Eclat y FP-Growth: alternativas a Apriori

**Cuellos de botella de Apriori:** (1) la generación de candidatos puede producir conjuntos enormes y (2) exige recorrer la base de datos varias veces [9].

- **Eclat** [8]: usa un diseño *vertical*: para cada producto guarda los identificadores de las transacciones donde aparece (*tidset*) y calcula el soporte con **intersecciones** de esos conjuntos, recorriendo en profundidad. Ejemplo con t1 = {a, b, c}, t2 = {a, b}, t3 = {a}: a → {t1, t2, t3}, b → {t1, t2}, c → {t1}; con `min_support` = 0.5 (mínimo 2 transacciones) c se descarta y el par {a, b} tiene tidset {t1, t2}, soporte 2/3.
- **FP-Growth** [7]: comprime los pedidos en un **árbol de patrones frecuentes (árbol FP)** y extrae los patrones sin generar candidatos, con solo dos recorridos de los datos.

## FP GROWTH(Frequent Pattern Growth)



In [ ]:
# FP-Growth con mlxtend
res_fp = fpgrowth(data, min_support=0.005, use_colnames=True)
print('Conjuntos frecuentes con FP-Growth:', len(res_fp), '| mismos que Apriori:', set(res_fp.itemsets) == set(freq_items.itemsets))

In [ ]:
reglas_fp = association_rules(res_fp, metric='lift', min_threshold=1)
print('Reglas con lift > 1:', len(reglas_fp))

In [ ]:
reglas_fp.sort_values('lift', ascending=False)[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10)

Para el mismo `min_support`, FP-Growth y Apriori deben entregar **los mismos conjuntos frecuentes** (se comprobó arriba); lo que cambia es el tiempo de cómputo.

## Apriori vs. FP-Growth: tiempos

Al no generar candidatos, FP-Growth suele escalar mejor que Apriori cuando hay muchas transacciones o soportes muy bajos [7]. Lo medimos en estos datos en lugar de suponerlo.

In [ ]:
import time
soportes = [0.005, 0.01, 0.02, 0.03]
t_apriori, t_fp = [], []
for s in soportes:
    t0 = time.perf_counter(); apriori(data, min_support=s, use_colnames=True); t_apriori.append((time.perf_counter() - t0) * 1000)
    t0 = time.perf_counter(); fpgrowth(data, min_support=s, use_colnames=True); t_fp.append((time.perf_counter() - t0) * 1000)
plt.plot(soportes, t_apriori, 'o-', label='Apriori')
plt.plot(soportes, t_fp, 's-', label='FP-Growth')
plt.xlabel('Soporte mínimo'); plt.ylabel('Tiempo (ms)'); plt.title('Tiempo de cómputo según el soporte mínimo'); plt.legend(); plt.show()

---
## 🧪 Refuerzo: soporte, confianza y lift calculados a mano

Antes de confiar en la librería, conviene entender de dónde salen los números. Con una canasta mínima de 5 compras calculamos las medidas con conteos y comprobamos que `mlxtend` [11] da lo mismo.

In [ ]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

mini = [['pan', 'leche'], ['pan', 'leche', 'huevos'], ['pan', 'huevos'], ['leche', 'huevos'], ['pan', 'leche', 'cafe']]
n = len(mini)
sop = lambda s: sum(set(s) <= set(t) for t in mini) / n

print('sop(pan)         =', sop(['pan']))
print('sop(pan, leche)  =', sop(['pan', 'leche']))
print('conf(pan→leche)  =', sop(['pan', 'leche']) / sop(['pan']))
print('lift(pan→leche)  =', sop(['pan', 'leche']) / (sop(['pan']) * sop(['leche'])))

te = TransactionEncoder()
M = pd.DataFrame(te.fit_transform(mini), columns=te.columns_)
reglas = association_rules(apriori(M, min_support=0.4, use_colnames=True), metric='confidence', min_threshold=0.5)
reglas[(reglas.antecedents == frozenset({'pan'})) & (reglas.consequents == frozenset({'leche'}))][['support', 'confidence', 'lift']]

### ✍️ Ejercicios
1. Calcule a mano la confianza y el lift de `huevos → leche` y verifique con `mlxtend`.
2. ¿Puede una regla tener confianza alta y lift menor que 1? Construya un ejemplo.
3. Si baja `min_support`, ¿qué pasa con la cantidad de conjuntos frecuentes y con el tiempo de cómputo? [5]

---
## 🧪 Refuerzo: Eclat desde cero: contar con intersecciones de conjuntos de transacciones

**Eclat** recorre los datos en **vertical**: para cada producto guarda el conjunto de identificadores de las transacciones donde aparece (*tidset*). El soporte de un conjunto de productos es el tamaño de la **intersección** de sus tidsets [8]. A continuación se programa una versión mínima (recursiva, en profundidad) y se comprueba que encuentra **exactamente los mismos conjuntos frecuentes** que Apriori.

In [ ]:
import numpy as np

def eclat(matriz, min_support):
    n = len(matriz)
    minimo = min_support * n
    tids = {p: set(np.flatnonzero(matriz[p].values)) for p in matriz.columns}
    tids = {p: t for p, t in tids.items() if len(t) >= minimo}
    resultado = {}

    def recorrer(prefijo, candidatos):
        for i, (p, t) in enumerate(candidatos):
            nuevo = prefijo + (p,)
            resultado[frozenset(nuevo)] = len(t) / n
            siguientes = []
            for q, tq in candidatos[i + 1:]:
                inter = t & tq                      # intersección de tidsets = soporte del par
                if len(inter) >= minimo:
                    siguientes.append((q, inter))
            if siguientes:
                recorrer(nuevo, siguientes)

    recorrer((), sorted(tids.items()))
    return resultado

res_eclat = eclat(data, 0.01)
res_apriori = apriori(data, min_support=0.01, use_colnames=True)
print('Conjuntos con Eclat:  ', len(res_eclat))
print('Conjuntos con Apriori:', len(res_apriori))
print('¿Mismos conjuntos y mismo soporte?', all(abs(res_eclat[s] - sop) < 1e-9 for s, sop in zip(res_apriori.itemsets, res_apriori.support)) and len(res_eclat) == len(res_apriori))

### ✍️ Ejercicios
1. Mida el tiempo de `eclat`, `apriori` y `fpgrowth` con `min_support = 0.005`. ¿Cuál gana en este conjunto de datos?
2. ¿Por qué el *tidset* de un conjunto de tres productos se obtiene intersectando solo dos tidsets?
3. Eclat usa memoria para guardar los tidsets. ¿Cuándo podría ser un problema? [8]

---
## ✅ Conclusiones y siguientes pasos

Con pedidos completos, podando los productos poco frecuentes y comparando tres algoritmos se ve que **la elección de `min_support` pesa más que la del algoritmo**: Apriori, Eclat y FP-Growth devuelven los mismos conjuntos frecuentes y solo difieren en el tiempo [7] [8]. Las reglas describen co-ocurrencia en carritos, no causalidad [10]; el lift ayuda a no confundir asociación con simple popularidad [6].

---
## 📚 Referencias

Fuentes adicionales para profundizar: [12], [13], [14].

**[1]** Chapman, P., Clinton, J., Kerber, R., Khabaza, T., Reinartz, T., Shearer, C., & Wirth, R. (2000). *CRISP-DM 1.0: Step-by-step data mining guide*. SPSS Inc. https://www.the-modeling-agency.com/crisp-dm.pdf

**[2]** Wirth, R., & Hipp, J. (2000). CRISP-DM: Towards a standard process model for data mining. *Proceedings of the 4th International Conference on the Practical Applications of Knowledge Discovery and Data Mining*, 29–39.

**[3]** Instacart. (2017). *The Instacart Online Grocery Shopping Dataset 2017* [Conjunto de datos]. https://www.instacart.com/datasets/grocery-shopping-2017

**[4]** Agrawal, R., Imieliński, T., & Swami, A. (1993). Mining association rules between sets of items in large databases. *Proceedings of the 1993 ACM SIGMOD International Conference on Management of Data*, 207–216. https://doi.org/10.1145/170035.170072

**[5]** Agrawal, R., & Srikant, R. (1994). Fast algorithms for mining association rules. *Proceedings of the 20th International Conference on Very Large Data Bases (VLDB)*, 487–499.

**[6]** Brin, S., Motwani, R., Ullman, J. D., & Tsur, S. (1997). Dynamic itemset counting and implication rules for market basket data. *Proceedings of the 1997 ACM SIGMOD International Conference on Management of Data*, 255–264.

**[7]** Han, J., Pei, J., & Yin, Y. (2000). Mining frequent patterns without candidate generation. *Proceedings of the 2000 ACM SIGMOD International Conference on Management of Data*, 1–12.

**[8]** Zaki, M. J. (2000). Scalable algorithms for association mining. *IEEE Transactions on Knowledge and Data Engineering, 12*(3), 372–390. https://doi.org/10.1109/69.846291

**[9]** Han, J., Kamber, M., & Pei, J. (2012). *Data Mining: Concepts and Techniques* (3.ª ed.). Morgan Kaufmann. https://doi.org/10.1016/C2009-0-61819-5

**[10]** Tan, P.-N., Steinbach, M., Karpatne, A., & Kumar, V. (2019). *Introduction to Data Mining* (2.ª ed.). Pearson.

**[11]** Raschka, S. (2018). MLxtend: Providing machine learning and data science utilities and extensions to Python's scientific computing stack. *Journal of Open Source Software, 3*(24), 638. https://doi.org/10.21105/joss.00638

**[12]** McKinney, W. (2010). Data structures for statistical computing in Python. *Proceedings of the 9th Python in Science Conference*, 56–61. https://doi.org/10.25080/Majora-92bf1922-00a

**[13]** Waskom, M. L. (2021). seaborn: statistical data visualization. *Journal of Open Source Software, 6*(60), 3021. https://doi.org/10.21105/joss.03021

**[14]** Hunter, J. D. (2007). Matplotlib: A 2D graphics environment. *Computing in Science & Engineering, 9*(3), 90–95. https://doi.org/10.1109/MCSE.2007.55